# Xử lý ảnh thiếu sáng cho phân loại biển báo GTSRB

**Bài tập lớn môn Xử lý ảnh - Nhóm 2.** Một notebook cho toàn bộ nghiên cứu: bảy phương pháp tăng sáng, ba bộ lọc khử nhiễu và hai thứ tự kết hợp. Chọn tham số trên validation, đánh giá subset và full-test, xuất bằng chứng có thể kiểm tra lại.

`reuse` dùng ba checkpoint và kết quả đối chứng đã xác minh để chạy khảo sát khử nhiễu; `fresh` huấn luyện từ đầu. Không tự huấn luyện khi thiếu checkpoint. Kết quả tái sử dụng được ghi rõ nguồn. Chỉ trạng thái `completed` cùng gói kết quả qua kiểm tra mới xác nhận hoàn tất; không gọi subset là full-test hoặc coi ảnh tối mô phỏng là camera đêm thực.

GTSRB tải từ [archive tác giả](https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/published-archive.html). Trọng số enhancer tải từ repo tác giả, khóa commit và SHA-256. Giữ giấy phép và ghi công; không coi việc tải là cấp phép lại. Zero-DCE chỉ dùng cho nghiên cứu/học thuật phi thương mại. Không cần API key trong notebook.

## 1. Lấy mã nguồn từ GitHub

Notebook tải mã từ [GitHub](https://github.com/kttt294/GTSRB-Low-Light-Preprocessing) tại một commit cố định, checkout đúng phiên bản rồi import các module; hash của 22 file Python/cấu hình được ghi vào kết quả để truy vết. Không lấy nhánh main mới nhất, không nhúng toàn bộ mã vào cell. Mở `src/` trên GitHub để đọc phần triển khai.

Một notebook tổ chức toàn bộ quy trình chuẩn bị dữ liệu, train/khôi phục, chọn tham số, đánh giá và đóng gói. Cần Internet để lấy mã, GTSRB, weights tác giả và ZIP kết quả dùng lại. Các đầu vào nằm trong `/tmp/gtsrb-clean`; chỉ kết quả và checkpoint được xuất vào `/kaggle/working`. Không cần API key trong notebook.

In [ ]:
import os, sys, json, hashlib, subprocess, shutil
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path('/tmp/gtsrb-clean')
PUBLIC = Path('/kaggle/working')
ROOT.mkdir(parents=True,exist_ok=True)
PUBLIC.mkdir(parents=True,exist_ok=True)

In [ ]:
SOURCE_REPOSITORY = 'https://github.com/kttt294/GTSRB-Low-Light-Preprocessing.git'
SOURCE_COMMIT = 'f3169402c1e2cdf584467b220881156535d44240'
SOURCE_FILES = (
    'src/__init__.py src/common.py src/data.py src/degradation.py src/enhancement.py src/experiment.py '
    'src/metrics.py src/model.py src/reporting.py src/training.py src/tools/__init__.py '
    'src/tools/public_sources.py src/tools/fresh_run.py src/tools/research_evidence.py '
    'src/tools/result_archive.py src/tools/checkpoint_restore.py src/tools/processing_pipeline.py '
    'src/tools/processing_study.py src/tools/baseline_input.py configs/resnet18.json '
    'configs/public_sources.json configs/processing_extension.json '
).split()

REPOSITORY = ROOT / 'repository'
if not REPOSITORY.exists():
    subprocess.run(['git', 'clone', '--no-checkout', SOURCE_REPOSITORY, str(REPOSITORY)], check=True)
subprocess.run(['git', '-C', str(REPOSITORY), 'checkout', '--detach', SOURCE_COMMIT], check=True)
SOURCE_HASHES = {}
for name in SOURCE_FILES:
    content = subprocess.check_output(['git', '-C', str(REPOSITORY), 'show', SOURCE_COMMIT + ':' + name])
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
    SOURCE_HASHES[name] = hashlib.sha256(content).hexdigest()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Source commit:', SOURCE_COMMIT)

## 2. Môi trường và cấu hình

Giữ bộ PyTorch/CUDA của Kaggle. Chỉ cài dependency còn thiếu. Nếu có hai GPU, các training seed và nhóm đánh giá được phân công độc lập giữa hai GPU; mỗi seed vẫn giữ batch size, optimizer và dữ liệu như cấu hình. Trọng số ba enhancer chỉ dùng suy luận.

Full-test tốn nhiều giờ. Notebook giải phóng cache sau khi cả ba classifier dùng xong một nhóm ảnh; không lưu toàn bộ cache vào Output. Không tự giảm số seed hoặc phương pháp khi sắp hết thời gian.

In [ ]:
import importlib.util, importlib.metadata
dependencies = {'einops':'einops','yaml':'PyYAML','pytorch_msssim':'pytorch-msssim','gdown':'gdown',
                'skimage':'scikit-image','sklearn':'scikit-learn','cv2':'opencv-python-headless'}
missing = [package for module,package in dependencies.items() if importlib.util.find_spec(module) is None]
try:
    importlib.metadata.version('opencv-python-headless')
except importlib.metadata.PackageNotFoundError:
    missing.append('opencv-python-headless')
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*dict.fromkeys(missing)],check=True)
import torch, numpy as np, pandas as pd
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Bật GPU và Internet trong Settings trước khi chạy.'
DEVICES = [f'cuda:{i}' for i in range(torch.cuda.device_count())]
CONFIG = json.loads((ROOT/'configs/resnet18.json').read_text())
TRAIN_MODE = 'reuse'  # 'fresh' để chạy toàn bộ từ đầu.
CHECKPOINT_ARCHIVE = None  # Optional: ZIP đã có trong phiên; None tải trực tiếp Output v1.
REQUIRE_BASELINE_REUSE = True  # Stop instead of rerunning completed baseline experiments.
RUN_ABLATION = TRAIN_MODE == 'fresh'
RUN_EXTENSION_FULL_TEST = True
assert TRAIN_MODE in {'fresh','reuse'}
if TRAIN_MODE == 'reuse':
    from src.tools.baseline_input import resolve_baseline
    CHECKPOINT_ARCHIVE = resolve_baseline(ROOT/'baseline_input',CHECKPOINT_ARCHIVE)
OUT = ROOT/('outputs/fresh' if TRAIN_MODE=='fresh' else 'outputs/reuse')
OUT.mkdir(parents=True,exist_ok=True)
NOTEBOOK_URL = 'https://www.kaggle.com/code/kieuthithutrang/gtsrb-low-light-enhancement-for-traffic-signs'
from src.common import write_json, environment
def status(stage, **extra):
    record = {'stage':stage,'notebook_url':NOTEBOOK_URL,'fresh_training':TRAIN_MODE=='fresh','extension_requested':True,**extra}
    write_json(PUBLIC/'RUN_STATUS.json',record)
    print('STAGE:',stage,flush=True)
status('environment_ready',devices=DEVICES)
print('PyTorch:',torch.__version__,'GPU:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 3. Tải và kiểm tra trọng số tác giả trước khi train

Kiểm tra tải được file, đúng checksum, đúng kiến trúc và suy luận được RGB 64×64. Nếu nguồn tác giả không truy cập được, dừng trước huấn luyện; không thay bằng trọng số khác hoặc lấy từ dataset private.

In [ ]:
from src.tools.public_sources import provision
from src.enhancement import Enhancer
UPSTREAM = provision(ROOT,ROOT/'configs/public_sources.json')
for method in ['zero_dce','retinexformer','diffusion']:
    enhancer = Enhancer(method,CONFIG['enhancers'][method],torch.device(DEVICES[0]))
    preview = enhancer(np.full((64,64,3),.15,dtype=np.float32),'preflight')
    assert preview.shape == (64,64,3) and np.isfinite(preview).all()
    print('Forward OK:',method,flush=True)
    del enhancer
torch.cuda.empty_cache()
write_json(OUT/'upstream_sources.json',UPSTREAM)
write_json(OUT/'source_hashes.json',SOURCE_HASHES)
status('upstream_verified')

## 4. GTSRB chính thức và chia dữ liệu

Nhãn train nằm trong 43 file `GT-*.csv` ngay trong ZIP ảnh train; nhãn test tải bằng gói GT test riêng. Chia train/validation theo class+track, seed 42, cách ly track có ảnh trùng pixel với test; crop ROI padding 5%, RGB 64×64. Chọn trước ID subset test để không chọn ví dụ theo kết quả.

In [ ]:
from src.data import download, prepare, audit, subset_rows
from src.common import read_csv
download(ROOT)
prepare(ROOT,CONFIG)
audit(ROOT)
rows = read_csv(ROOT/'data/manifests/gtsrb_samples.csv')
display(pd.DataFrame(rows).groupby('split').size().rename('Số ảnh').to_frame())
assert sum(r['split']=='test' for r in rows) == 12630
assert sum(r['split']=='train' for r in rows) == 31350
assert sum(r['split']=='validation' for r in rows) == 7829
write_json(OUT/'predeclared_subset_ids.json',[r['sample_id'] for r in subset_rows(rows,'test',10,42)])
status('dataset_verified')

## 5. Khôi phục hoặc huấn luyện ba ResNet-18

`reuse`: tải ZIP trực tiếp từ Output của phiên v1 của chính notebook, không tạo Dataset và không cần API key. Có thể điền CHECKPOINT_ARCHIVE để dùng ZIP đã có. Kiểm tra checksum gói gốc, sửa riêng đường dẫn supplemental bị trùng và đối chiếu toàn bộ manifest với gói đã kiểm chứng; không thay dự đoán hay checkpoint. Sau đó kiểm tra ba seed, manifest và cấu hình CNN; chỉ đọc checkpoint bằng weights_only, không chạy mã trong ZIP. Selection và dự đoán chỉ được dùng lại khi cùng mã, dữ liệu và checkpoint. `fresh`: ImageNet, tinh chỉnh ảnh gốc, seed 11/22/33, tối đa 20 epoch; chọn checkpoint bằng Macro-F1 validation clean. Sao lưu checkpoint ngay sau bước này.

In [ ]:
from src.tools.fresh_run import train_fresh
from src.common import digest
if TRAIN_MODE == 'fresh':
    status('training',devices=DEVICES)
    CHECKPOINTS = train_fresh(ROOT,CONFIG,OUT/'train',DEVICES)
else:
    from src.tools.checkpoint_restore import restore
    CHECKPOINTS, RESTORED = restore(ROOT,CHECKPOINT_ARCHIVE,OUT,CONFIG)
    shutil.copy2(ROOT/'baseline_input/input_provenance.json',OUT/'train/input_provenance.json')
    print(RESTORED)
    if REQUIRE_BASELINE_REUSE:
        assert RESTORED['selection_reusable'] and set(RESTORED['baseline_scopes'])=={'subset','fulltest'} and RESTORED['ablation_reusable'], 'Baseline artifacts cannot be reused exactly; stopping before any baseline retraining or reevaluation.'
write_json(OUT/'checkpoint_hashes.json',{p.name:digest(p) for p in CHECKPOINTS})
import zipfile
backup = PUBLIC/'gtsrb-checkpoints.zip'
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    hashes={}
    for p in CHECKPOINTS:
        name='05_training/'+p.name;z.write(p,name);hashes[name]=digest(p)
    z.writestr('SHA256.json',json.dumps(hashes,indent=2))
status('classifiers_ready',checkpoint_backup=backup.name)

## 6. Chọn phương pháp trên validation

1.290 ảnh validation, 3 mức suy giảm và 3 noise seed, cùng ba classifier. Chọn cấu hình Gamma/CLAHE/MSR và pipeline bằng Macro-F1 validation; chưa xem test. Cache giải phóng sau khi ba checkpoint dùng xong.

In [ ]:
from src.experiment import tune, freeze
status('validation_selection')
if (OUT/'validation/selection.json').exists():
    selection = json.loads((OUT/'validation/selection.json').read_text())
else:
    selection = tune(ROOT,CONFIG,CHECKPOINTS,torch.device(DEVICES[0]),OUT/'validation',per_class=30,release_cache=True)
torch.cuda.empty_cache()
display(pd.DataFrame(selection['scores']))
print('Pipeline chọn trước test:',selection['chosen_method'])
status('validation_completed',selected_method=selection['chosen_method'])

## 7. Subset 430 ảnh: kiểm tra giao thức và phân tích lỗi

219 lượt, đủ 43 lớp (10 ảnh/lớp). Tính lại metric từ dự đoán, F1 từng lớp, ma trận nhầm lẫn và các trường hợp được giúp/bị làm sai. Đây là subset; không gọi là full-test. Xuất gói subset trước để vẫn giữ được bằng chứng nếu phiên Kaggle bị ngắt trong các bước dài sau đó.

In [ ]:
from src.tools.fresh_run import parallel_evaluate, ablation
from src.tools.research_evidence import analyze, audited_runs
from src.tools.result_archive import pack, aggregate
subset_protocol = OUT/'protocol_subset.json'
status('subset_evaluation')
if (OUT/'subset/summary.csv').exists():
    audited_runs(OUT/'subset')
else:
    if not subset_protocol.exists():freeze(ROOT,OUT/'validation/selection.json',subset_protocol,per_class=10)
    parallel_evaluate(ROOT,subset_protocol,OUT/'subset',DEVICES)
if not (OUT/'subset/error_analysis/audit.json').exists():analyze(OUT/'subset',OUT/'subset/error_analysis')
display(pd.DataFrame(aggregate(read_csv(OUT/'subset/summary.csv')))[['condition','method','accuracy_mean','macro_f1_mean','macro_f1_sd']])
if not (PUBLIC/'gtsrb-subset-results.zip').exists():pack(OUT/'subset',PUBLIC/'gtsrb-subset-results.zip',analysis=OUT/'subset/error_analysis',source=ROOT)
status('subset_completed',prediction_rows=94170)

## 8. Ablation trên subset

Tách Gamma-only, Gamma+Gaussian, Gamma+Poisson và Gamma+cả hai, giữ nguyên checkpoint và tham số đã chọn. Có 288 lượt. Không dùng kết quả ablation để chọn lại pipeline.

In [ ]:
if RUN_ABLATION:
    status('ablation')
    ablation(ROOT,CONFIG,selection,CHECKPOINTS,torch.device(DEVICES[0]),OUT/'ablation')
    display(pd.DataFrame(read_csv(OUT/'ablation/summary.csv')).head())
else:
    print('Dùng lại ablation đã kiểm tra.' if (OUT/'ablation/summary.csv').exists() else 'Ablation không được yêu cầu.')
torch.cuda.empty_cache()
status('ablation_completed' if RUN_ABLATION else 'ablation_not_requested')

## 9. Full-test chính thức 12.630 ảnh

Khóa cùng selection rồi đánh giá đủ 219 lượt, tổng cộng 2.765.970 dự đoán. Các nhóm condition/noise/method độc lập được phân cho GPU khả dụng; cả ba checkpoint nhận cùng ảnh tăng cường trong mỗi nhóm. Cache được tái tạo đúng seed và giải phóng sau nhóm, giữ thông tin hash và dự đoán. Nếu chưa xong bước này, không coi bảng subset là full-test.

In [ ]:
full_protocol = OUT/'protocol_fulltest.json'
status('fulltest_evaluation')
if (OUT/'fulltest/summary.csv').exists():
    audited_runs(OUT/'fulltest')
else:
    if not full_protocol.exists():freeze(ROOT,OUT/'validation/selection.json',full_protocol)
    parallel_evaluate(ROOT,full_protocol,OUT/'fulltest',DEVICES)
if not (OUT/'fulltest/error_analysis/audit.json').exists():analyze(OUT/'fulltest',OUT/'fulltest/error_analysis')
full_aggregate = pd.DataFrame(aggregate(read_csv(OUT/'fulltest/summary.csv')))
display(full_aggregate[['condition','method','accuracy_mean','macro_f1_mean','macro_f1_sd']])
status('fulltest_verified',runs=219,images_per_run=12630,prediction_rows=2765970)
baseline_backup = PUBLIC/'gtsrb-baseline-results.zip'
if not baseline_backup.exists():
    pack(OUT/'fulltest',baseline_backup,analysis=OUT/'fulltest/error_analysis',source=ROOT,
         supplemental=OUT/'ablation',subset=OUT/'subset')
print('Baseline full-test đã được sao lưu trước khi chạy mở rộng:',baseline_backup.name)

## 10. Mở rộng: khử nhiễu và thứ tự xử lý

Gaussian, bilateral và Non-local Means: hai cấu hình mỗi bộ lọc. Chọn độ mạnh trên 1.290 ảnh validation rồi **giữ cùng tham số** khi so hai thứ tự với Gamma và CLAHE. Tổng cộng 6 ứng viên độc lập và 12 chuỗi kết hợp cần đánh giá thêm trên validation. Không thay CNN, mức tối, noise seed hoặc test ID.

Subset chạy mọi họ phương pháp. Validation chọn một cặp thứ tự theo điểm trung bình hai thứ tự, kèm đối chứng độc lập, bộ lọc tốt nhất và ứng viên tổng thể tốt nhất cho full-test. Khóa cả hai giao thức trước khi đọc test; không chọn theo subset test. NLM chuyển RGB sang uint8 BGR theo OpenCV và ghi lượng tử hóa vào provenance. Pipeline không đọc nhãn hay ảnh gốc; bộ đánh giá mới đọc để tính metric.

In [ ]:
from src.tools import processing_study as extension
EXT = OUT/'processing_extension'
status('extension_validation')
extension_selection = extension.tune(ROOT,OUT/'validation/selection.json',CHECKPOINTS,
    ROOT/'configs/processing_extension.json',torch.device(DEVICES[0]),EXT/'validation')
display(pd.DataFrame(extension_selection['scores'])[['method','category','macro_f1','latency_ms']])
print('Full-test methods locked from validation:',extension_selection['fulltest_methods'])
extension.freeze(ROOT,EXT/'validation/selection.json',EXT/'protocol_subset.json',per_class=10)
extension.freeze(ROOT,EXT/'validation/selection.json',EXT/'protocol_fulltest.json',per_class=None)
for scope in (['subset','fulltest'] if RUN_EXTENSION_FULL_TEST else ['subset']):
    status('extension_'+scope)
    extension.evaluate(ROOT,EXT/('protocol_'+scope+'.json'),EXT/scope,torch.device(DEVICES[0]),
        baseline_output=OUT/scope)
    if not (EXT/scope/'error_analysis/audit.json').exists():
        extension.analyze(EXT/scope,EXT/scope/'error_analysis')
    display(pd.DataFrame(aggregate(read_csv(EXT/scope/'summary.csv')))[
        ['condition','method','accuracy_mean','macro_f1_mean','macro_f1_sd','psnr_mean_mean','ssim_mean_mean']])
    if scope=='subset' and not (PUBLIC/'gtsrb-processing-subset-results.zip').exists():
        pack(EXT/scope,PUBLIC/'gtsrb-processing-subset-results.zip',
             analysis=EXT/scope/'error_analysis',include_checkpoints=False)
torch.cuda.empty_cache()
status('extension_verified',extension_fulltest_completed=RUN_EXTENSION_FULL_TEST)

## 11. Xuất gói kết quả dễ đọc

Mở `READ_ME_FIRST.md` trong ZIP trước. Bảng dự đoán test gộp vào một CSV với run_id/method/condition/seed; confusion matrix gộp vào NPZ, có bản kê SHA-256. Validation, subset và ablation nằm riêng, không trộn với full-test. Không đóng gói ảnh gốc, cache hoặc trọng số enhancer tác giả.

Gói subset có checkpoint để khôi phục nếu phiên bị ngắt. Sau khi full-test hoàn tất, checkpoint chỉ nằm trong gói đầy đủ, tránh giữ hai bản trùng nhau. Kết quả ảnh đêm thực chưa được tạo vì chưa có dữ liệu thực gán nhãn tương thích.

In [ ]:
final_zip = PUBLIC/'gtsrb-notebook-results.zip'
staged_zip = PUBLIC/'.gtsrb-final-export.zip'
if staged_zip.exists():staged_zip.unlink()
pack(OUT/'fulltest',staged_zip,analysis=OUT/'fulltest/error_analysis',source=ROOT,
     supplemental=OUT/'ablation',subset=OUT/'subset',extension=EXT)
staged_zip.replace(final_zip)
import zipfile
with zipfile.ZipFile(final_zip) as z:
    assert z.testzip() is None
    print('ZIP:',final_zip.name,'files:',len(z.namelist()),'MB:',round(final_zip.stat().st_size/1e6,1))
    print('Top-level:',sorted({n.split('/')[0] for n in z.namelist()}))
# Replace only our own explicitly named subset ZIP with a tables-only version.
tables_only = PUBLIC/'.gtsrb-subset-tables.zip'
if tables_only.exists():tables_only.unlink()
pack(OUT/'subset',tables_only,analysis=OUT/'subset/error_analysis',include_checkpoints=False)
tables_only.replace(PUBLIC/'gtsrb-subset-results.zip')
# Remove our explicitly named interim backups only after final ZIP verification.
for name in ['gtsrb-checkpoints.zip','gtsrb-baseline-results.zip','gtsrb-processing-subset-results.zip']:
    path = PUBLIC/name
    if path.exists():path.unlink()
status('completed',final_archive=final_zip.name,archive_sha256=digest(final_zip),
       fulltest_prediction_rows=2765970,real_lowlight_evaluated=False,extension_fulltest_completed=RUN_EXTENSION_FULL_TEST)
print('COMPLETE — tải gtsrb-notebook-results.zip ở Output của notebook.')

## Tài liệu và phạm vi

- GTSRB: J. Stallkamp, M. Schlipsing, J. Salmen, C. Igel, *The German Traffic Sign Recognition Benchmark: A multi-class classification competition*, IJCNN 2011.
- Zero-DCE: C. Guo et al., CVPR 2020.
- Retinexformer: Y. Cai et al., ICCV 2023.
- Wavelet-based Diffusion LLIE: H. Jiang et al., ACM TOG 2023.

Ảnh crop và suy giảm tổng hợp phục vụ nghiên cứu. PSNR/SSIM đo so ảnh gốc trên cùng crop; không thay thế hiệu năng phân loại. Độ lệch chuẩn qua ba checkpoint không phải khoảng tin cậy cho camera thực. Không có phát hiện biển báo toàn cảnh hay đánh giá ảnh đêm thực trong lần chạy này.